# Agentic AI-Based Dynamic Tariff Optimization for EV Charging Networks

**Author:** Ashish | B.Tech Production and Industrial Engineering | Enrollment: 23119004  
**Competition:** Open Project 2026 | Society of Business  
**Date:** June 2026

---

### Pipeline Overview
1. Setup & Data Loading
2. Preprocessing & Aggregation  
3. Feature Engineering (93 features)
4. Demand Prediction (Optuna + CatBoost R2=0.861)
5. Tariff Pricing Agent (9 strategies + spatial equilibrium)
6. Monitoring Agent (Q-Learning RL)
7. Results, Figures & Assumptions

### How to Run
- **Google Colab:** Upload the notebook, then click **Runtime → Run All**. The notebook auto-downloads data from GitHub (first run ~20s). No manual upload needed.
- **Local Machine:** Place this notebook in the project root folder alongside `sz_districts/`, then Run All

### Required Data Files
Create a zip called `project_data.zip` containing:
- `sz_districts/` folder (with all 9 CSV files)
- `acndata_sessions.json.xlsx` (ACN dataset)


## 1. Setup & Imports

**Cell 1** installs required packages on Colab (skipped on local).  
**Cell 2** auto-downloads data from GitHub on Colab, or auto-detects local project directory.


In [ ]:
# --- Detect environment ---
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Environment: {"Google Colab" if IN_COLAB else "Local Machine"}')

# --- Install packages (Colab only) ---
if IN_COLAB:
    print('Installing required packages...')
    !pip install -q lightgbm xgboost catboost optuna openpyxl
    print('Packages installed.')
else:
    print('Local machine detected -- skipping package install.')
    print('Make sure you have: pip install numpy pandas scipy scikit-learn lightgbm xgboost catboost optuna matplotlib openpyxl')


In [ ]:
# --- Data Loading ---
# Colab: upload project_data.zip (containing sz_districts/ and acndata_sessions.json.xlsx)
# Local: auto-detects the project directory containing sz_districts/

import os, sys, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy.stats import skew
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.cluster import KMeans
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
np.random.seed(42)

if IN_COLAB:
    # --- Colab: auto-download data from GitHub release ---
    BASE_DIR = '/content'
    _data_dir = os.path.join(BASE_DIR, 'sz_districts')
    _acn_file = os.path.join(BASE_DIR, 'acndata_sessions.json.xlsx')
    
    if not os.path.exists(_data_dir) or not os.path.exists(_acn_file):
        print('Downloading project_data.zip from GitHub release...')
        github_url = 'https://github.com/ashishexee/ev-charging-data/releases/download/v1.0/project_data.zip'
        !wget -q "{github_url}" -O /content/project_data.zip
        import zipfile
        with zipfile.ZipFile('/content/project_data.zip', 'r') as zf:
            zf.extractall('/content')
        print('Data downloaded and extracted to /content/')
        if not os.path.exists(_data_dir):
            raise RuntimeError(f'Data extraction failed. {github_url} may be unreachable or the zip structure changed.')
    else:
        print('Data already exists, skipping download.')
else:
    # --- Local: auto-detect project directory ---
    BASE_DIR = os.getcwd()
    if not os.path.exists(os.path.join(BASE_DIR, 'sz_districts')):
        BASE_DIR = os.path.dirname(BASE_DIR)
    if not os.path.exists(os.path.join(BASE_DIR, 'sz_districts')):
        for candidate in [os.path.expanduser('~/socbiz'), os.path.expanduser('~/Desktop/socbiz')]:
            if os.path.exists(os.path.join(candidate, 'sz_districts')):
                BASE_DIR = candidate
                break

DATA_DIR = os.path.join(BASE_DIR, 'sz_districts')
ACN_FILE = os.path.join(BASE_DIR, 'acndata_sessions.json.xlsx')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
METRICS_DIR = os.path.join(OUTPUT_DIR, 'metrics')
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')
CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints_final')

RS_BASE_TARIFF = 15.0
PRICE_MIN, PRICE_MAX = 4.0, 30.0
CAPACITY_UTIL_MAX = 1.0
SUBSTITUTION_RATE = 0.25
HORIZON = 6
N_CLUSTERS = 8

for d in [OUTPUT_DIR, METRICS_DIR, FIGURES_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Auto-detect GPU (works on both Colab and local) ---
HAS_GPU = False
LGB_DEVICE = 'cpu'
XGB_DEVICE = 'cpu'
XGB_TREE_METHOD = 'hist'
CAT_TASK_TYPE = 'CPU'

try:
    import torch
    if torch.cuda.is_available():
        HAS_GPU = True
except ImportError:
    pass

# Test LightGBM GPU (requires OpenCL)
try:
    _m = lgb.LGBMRegressor(device='gpu', n_estimators=1, verbose=-1)
    _m.fit(np.random.rand(10, 5), np.random.rand(10))
    LGB_DEVICE = 'gpu'
except Exception:
    LGB_DEVICE = 'cpu'

if HAS_GPU:
    XGB_DEVICE = 'cuda'
    CAT_TASK_TYPE = 'GPU'

print(f'\nProject dir: {BASE_DIR}')
print(f'Data found: {os.path.exists(DATA_DIR)}')
print(f'ACN found: {os.path.exists(ACN_FILE)}')
print(f'GPU: {HAS_GPU} | LightGBM: {LGB_DEVICE} | XGBoost: {XGB_DEVICE} | CatBoost: {CAT_TASK_TYPE}')
print(f'LightGBM {lgb.__version__} | XGBoost {xgb.__version__} | CatBoost available')


## 2. Data Loading & Preprocessing

**Data Sources:**
- **Shenzhen UrbanEV Dataset:** 247 zones, 720 hours (Jun-Jul 2022), 57 dynamic + 190 fixed pricing zones
- **ACN-Data:** 14,999 sessions from Caltech campus (Apr-Dec 2018), used for cross-validation

**Preprocessing Steps:**
1. Load raw CSV files (occupancy, volume, price, duration, adj, distance, info)
2. Compute utilization rate = occupancy / capacity
3. Aggregate 5-min intervals to hourly (12 timesteps per hour)
4. Compute spatial features (neighbor averages, util gradient, demand-supply ratio)
5. KMeans clustering on 24-hour utilization profiles (k=8)
6. Temporal train/val/test split: 60%/20%/20% (timesteps 0-431 / 432-575 / 576-713)


In [ ]:
t_start_global = time.time()

print('[1] Loading raw data...')
assert os.path.exists(DATA_DIR), f'Data directory not found: {DATA_DIR}. Upload project_data.zip on Colab or place sz_districts/ next to notebook on local.'

time_df = pd.read_csv(os.path.join(DATA_DIR, 'time.csv'))
occ_df = pd.read_csv(os.path.join(DATA_DIR, 'occupancy.csv'))
vol_df = pd.read_csv(os.path.join(DATA_DIR, 'volume.csv'))
prc_df = pd.read_csv(os.path.join(DATA_DIR, 'price.csv'))
info_df = pd.read_csv(os.path.join(DATA_DIR, 'information.csv'))
adj_df = pd.read_csv(os.path.join(DATA_DIR, 'adj.csv'))
dist_df = pd.read_csv(os.path.join(DATA_DIR, 'distance.csv'))

zone_cols = [c for c in occ_df.columns if c != 'timestamp']
info_df['grid'] = info_df['grid'].astype(str)
info_idx = info_df.set_index('grid')

capacity_map = info_idx['count'].to_dict()
dynamic_pricing_map = info_idx['dynamic_pricing'].to_dict()
cbd_map = info_idx['CBD'].to_dict()

n_ts, n_zones = occ_df.shape[0], len(zone_cols)

occ_vals = occ_df[zone_cols].values.astype(float)
vol_vals = vol_df[zone_cols].values.astype(float)
prc_vals = prc_df[zone_cols].values.astype(float)
cap_arr = np.array([capacity_map.get(z, np.nan) for z in zone_cols], dtype=float)
cap_median = np.nanmedian(cap_arr)
cap_arr = np.where(np.isnan(cap_arr), cap_median, cap_arr)

with np.errstate(divide='ignore', invalid='ignore'):
    util_rate = np.divide(occ_vals, cap_arr[np.newaxis, :])
    util_rate = np.where(np.isinf(util_rate), 0, util_rate)
    util_rate = np.where(np.isnan(util_rate), 0, util_rate)

timestamps = pd.to_datetime(
    time_df['year'].astype(str) + '-' + time_df['month'].astype(str) + '-' +
    time_df['day'].astype(str) + ' ' + time_df['hour'].astype(str) + ':' +
    time_df['minute'].astype(str) + ':' + time_df['second'].astype(str)
)
hours = timestamps.dt.hour.values
dayofweek = timestamps.dt.dayofweek.values
months = timestamps.dt.month.values
is_weekend = (dayofweek >= 5).astype(int)
period = np.zeros(n_ts, dtype=int)
peak_mask = ((hours >= 7) & (hours <= 10)) | ((hours >= 17) & (hours <= 20))
shoulder_mask = (hours >= 11) & (hours <= 16)
period[peak_mask] = 2
period[shoulder_mask] = 1

print('[2] Aggregating to hourly...')
n_hours = n_ts // 12
hourly_util = util_rate[:n_hours * 12].reshape(n_hours, 12, n_zones).mean(axis=1)
hourly_vol = vol_vals[:n_hours * 12].reshape(n_hours, 12, n_zones).mean(axis=1)
hourly_prc = prc_vals[:n_hours * 12].reshape(n_hours, 12, n_zones).mean(axis=1)

print('[3] Computing spatial features...')
adj_matrix = adj_df.values[:, 1:].astype(float)
dist_matrix = dist_df.values[:, 1:].astype(float)

neighbor_avg_util = np.zeros_like(hourly_util)
for i in range(n_zones):
    nb = np.where(adj_matrix[i] > 0)[0]
    if len(nb) > 0:
        neighbor_avg_util[:, i] = hourly_util[:, nb].mean(axis=1)

util_ma_1h_raw = np.zeros_like(util_rate)
util_std_1h_raw = np.zeros_like(util_rate)
for i in range(n_zones):
    s = pd.Series(util_rate[:, i])
    util_ma_1h_raw[:, i] = s.rolling(window=12, min_periods=1).mean().values
    util_std_1h_raw[:, i] = s.rolling(window=12, min_periods=1).std().fillna(0).values

hourly_nbr = neighbor_avg_util
hourly_ma1 = util_ma_1h_raw[:n_hours * 12].reshape(n_hours, 12, n_zones).mean(axis=1)
hourly_std = util_std_1h_raw[:n_hours * 12].reshape(n_hours, 12, n_zones).mean(axis=1)

A_float = adj_matrix.astype(np.float64)
util_gradient = np.zeros_like(hourly_util)
for z in range(n_zones):
    nb = np.where(A_float[z] > 0)[0]
    if len(nb) > 0:
        util_gradient[:, z] = hourly_util[:, z] - np.mean(hourly_util[:, nb], axis=1)
demand_supply = hourly_vol / (cap_arr.reshape(1, -1) + 1e-6)

print('[4] KMeans clustering (k=8)...')
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
zone_cluster = kmeans.fit_predict(hourly_util.T)
is_dynamic = np.array([dynamic_pricing_map.get(z, 0) for z in zone_cols], dtype=float)

hourly_time = pd.DataFrame({
    'hour': hours[::12][:n_hours],
    'day_of_week': dayofweek[::12][:n_hours],
    'is_weekend': is_weekend[::12][:n_hours],
    'period': period[::12][:n_hours],
    'hour_sin': np.sin(2 * np.pi * hours[::12][:n_hours] / 24),
    'hour_cos': np.cos(2 * np.pi * hours[::12][:n_hours] / 24),
    'dow_sin': np.sin(2 * np.pi * dayofweek[::12][:n_hours] / 7),
    'dow_cos': np.cos(2 * np.pi * dayofweek[::12][:n_hours] / 7),
    'month': months[::12][:n_hours],
})

zone_meta = pd.DataFrame({
    'zone_id': zone_cols,
    'capacity': cap_arr,
    'is_dynamic_pricing': [dynamic_pricing_map.get(z, 0) for z in zone_cols],
    'is_cbd': [cbd_map.get(z, 0) for z in zone_cols],
})

Z = n_zones
train_end = int(n_hours * 0.6)
val_end = int(n_hours * 0.8)

print(f'  Zones: {Z}, Hours: {n_hours}')
print(f'  Train: 0-{train_end}, Val: {train_end}-{val_end}, Test: {val_end}-{n_hours}')
print(f'  Dynamic zones: {int(is_dynamic.sum())}, Fixed: {int((1-is_dynamic).sum())}')

np.savez(os.path.join(CKPT_DIR, 'data.npz'),
         hourly_util=hourly_util, hourly_vol=hourly_vol, hourly_prc=hourly_prc,
         hourly_nbr=hourly_nbr, hourly_ma1=hourly_ma1, hourly_std=hourly_std,
         capacity=cap_arr, adj_matrix=adj_matrix,
         util_gradient=util_gradient, demand_supply=demand_supply,
         zone_cluster=zone_cluster, is_dynamic=is_dynamic,
         train_end=train_end, val_end=val_end, HORIZON=HORIZON, n_hours=n_hours)
hourly_time.to_csv(os.path.join(CKPT_DIR, 'hourly_time.csv'), index=False)
zone_meta.to_csv(os.path.join(CKPT_DIR, 'zone_meta.csv'), index=False)
print('  Checkpoint saved.')


## 3. Feature Engineering (93 Features)

**Feature Categories:**
1. **Target Encoding (16 of top-20 most important):** zone_mean, zone_std, zone_median, zone_p25, zone_p75, zone_skew, zone_peak_hour, zone_hour_mean, zone_hour_std, zone_dow_mean, cluster_mean, cluster_hour_mean, zone_hour_rank, zone_hour_zscore, expanding_mean (all computed from TRAINING data only to prevent leakage)
2. **Temporal (Fourier harmonics up to 4th order):** hour_sin/cos (1st-4th), dow_sin/cos, month_sin/cos, week_sin/cos, is_weekend, period
3. **Autoregressive (extended lags):** lag_1,2,3,6,12,24,48,72,168,336; diff_1,6,24,168; accel_6
4. **Rolling Statistics (7 windows):** roll_mean/std/min/max for windows 3,6,12,24,48,72,168
5. **Interaction Features:** util_x_price, util_x_neighbor, gradient_x_util, hour_x_weekend, lag24_x_capacity, zone_mean_x_hour
6. **Current State:** current_util, current_vol, current_prc, neighbor_util, ma_1h, std_1h, util_gradient, demand_supply_ratio
7. **Zone Properties:** capacity, is_dynamic, zone_cluster


In [ ]:
print('=' * 70)
print('FEATURE ENGINEERING - 93 Features')
print('=' * 70)

print('\n[1] Computing target encoding from training data ONLY...')
train_util = hourly_util[:train_end, :]
zone_mean = train_util.mean(axis=0)
zone_std = train_util.std(axis=0)
zone_median = np.median(train_util, axis=0)
zone_p25 = np.percentile(train_util, 25, axis=0)
zone_p75 = np.percentile(train_util, 75, axis=0)
zone_skew_arr = np.array([skew(train_util[:, z]) for z in range(Z)])

hourly_avg_for_peak = np.zeros((24, Z))
for h in range(24):
    mask = hourly_time.iloc[:train_end]['hour'].values == h
    if mask.sum() > 0:
        hourly_avg_for_peak[h, :] = train_util[mask, :].mean(axis=0)
zone_peak_hour = np.argmax(hourly_avg_for_peak, axis=0)

zone_hour_mean = np.zeros((24, Z))
zone_hour_std = np.zeros((24, Z))
for h in range(24):
    mask = hourly_time.iloc[:train_end]['hour'].values == h
    if mask.sum() > 0:
        zone_hour_mean[h, :] = train_util[mask, :].mean(axis=0)
        zone_hour_std[h, :] = train_util[mask, :].std(axis=0)

zone_dow_mean = np.zeros((7, Z))
for dow in range(7):
    mask = hourly_time.iloc[:train_end]['day_of_week'].values == dow
    if mask.sum() > 0:
        zone_dow_mean[dow, :] = train_util[mask, :].mean(axis=0)

n_clusters = len(np.unique(zone_cluster))
cluster_mean = np.zeros(n_clusters)
cluster_hour_mean = np.zeros((24, n_clusters))
for c in range(n_clusters):
    cz = np.where(zone_cluster == c)[0]
    cluster_mean[c] = train_util[:, cz].mean()
    for h in range(24):
        mask = hourly_time.iloc[:train_end]['hour'].values == h
        if mask.sum() > 0:
            cluster_hour_mean[h, c] = train_util[mask, :][:, cz].mean()

zone_hour_rank = np.zeros((n_hours, Z))
zone_hour_zscore = np.zeros((n_hours, Z))
for t in range(n_hours):
    row = hourly_util[t, :]
    zone_hour_rank[t, :] = np.argsort(np.argsort(row)) / max(Z - 1, 1)
    mu, sigma = row.mean(), row.std()
    zone_hour_zscore[t, :] = (row - mu) / max(sigma, 1e-8)

expanding_mean = np.zeros_like(hourly_util)
for z in range(Z):
    cumsum = np.cumsum(hourly_util[:, z])
    cumcount = np.arange(1, n_hours + 1)
    expanding_mean[:, z] = (cumsum - hourly_util[:, z]) / np.maximum(cumcount - 1, 1)

print('  Target encoding computed.')

print('\n[2] Building feature matrix (takes ~3-5 min)...')
t0 = time.time()

records = []
for z in range(Z):
    u = hourly_util[:, z]
    v = hourly_vol[:, z]
    p = hourly_prc[:, z]
    nbr = hourly_nbr[:, z]
    ma1 = hourly_ma1[:, z]
    std1 = hourly_std[:, z]
    ug = util_gradient[:, z]
    ds = demand_supply[:, z]
    zc = int(zone_cluster[z])
    iz = int(is_dynamic[z])
    for t in range(72, n_hours - HORIZON):
        row = hourly_time.iloc[t]
        h = int(row['hour'])
        dow = int(row['day_of_week'])
        lags = [u[t - k] if t - k >= 0 else 0 for k in [1, 2, 3, 6, 12, 24, 48, 72, 168, 336]]
        w3 = u[max(0, t - 3):t + 1]
        w6 = u[max(0, t - 6):t + 1]
        w12 = u[max(0, t - 12):t + 1]
        w24 = u[max(0, t - 24):t + 1]
        w48 = u[max(0, t - 48):t + 1]
        w72 = u[max(0, t - 72):t + 1]
        w168 = u[max(0, t - 168):t + 1]
        records.append({
            'zone': z, 'timestep': t,
            'hour': h, 'day_of_week': dow, 'is_weekend': row['is_weekend'],
            'period': row['period'],
            'hour_sin': row['hour_sin'], 'hour_cos': row['hour_cos'],
            'hour_sin_2': np.sin(4 * np.pi * h / 24),
            'hour_cos_2': np.cos(4 * np.pi * h / 24),
            'hour_sin_3': np.sin(6 * np.pi * h / 24),
            'hour_cos_3': np.cos(6 * np.pi * h / 24),
            'hour_sin_4': np.sin(8 * np.pi * h / 24),
            'hour_cos_4': np.cos(8 * np.pi * h / 24),
            'dow_sin': row['dow_sin'], 'dow_cos': row['dow_cos'],
            'dow_sin_2': np.sin(4 * np.pi * dow / 7),
            'dow_cos_2': np.cos(4 * np.pi * dow / 7),
            'month_sin_1': np.sin(2 * np.pi * int(row['month']) / 12),
            'month_cos_1': np.cos(2 * np.pi * int(row['month']) / 12),
            'month_sin_2': np.sin(4 * np.pi * int(row['month']) / 12),
            'month_cos_2': np.cos(4 * np.pi * int(row['month']) / 12),
            'week_sin_1': np.sin(2 * np.pi * t / 168),
            'week_cos_1': np.cos(2 * np.pi * t / 168),
            'week_sin_2': np.sin(4 * np.pi * t / 168),
            'week_cos_2': np.cos(4 * np.pi * t / 168),
            'lag_1': lags[0], 'lag_2': lags[1], 'lag_3': lags[2],
            'lag_6': lags[3], 'lag_12': lags[4], 'lag_24': lags[5],
            'lag_48': lags[6], 'lag_72': lags[7],
            'lag_168': lags[8], 'lag_336': lags[9],
            'diff_1': u[t] - lags[0], 'diff_6': u[t] - lags[3],
            'diff_24': u[t] - lags[5], 'diff_168': u[t] - lags[8],
            'accel_6': (u[t] - lags[3]) - (lags[3] - u[max(0, t - 12)]),
            'roll_mean_3': np.mean(w3), 'roll_std_3': np.std(w3) if len(w3) > 1 else 0,
            'roll_min_3': np.min(w3), 'roll_max_3': np.max(w3),
            'roll_mean_6': np.mean(w6), 'roll_std_6': np.std(w6) if len(w6) > 1 else 0,
            'roll_mean_12': np.mean(w12), 'roll_mean_24': np.mean(w24),
            'roll_range_12': np.max(w12) - np.min(w12),
            'roll_std_12': np.std(w12) if len(w12) > 1 else 0,
            'roll_min_12': np.min(w12), 'roll_max_12': np.max(w12),
            'roll_mean_48': np.mean(w48), 'roll_std_48': np.std(w48) if len(w48) > 1 else 0,
            'roll_min_48': np.min(w48), 'roll_max_48': np.max(w48),
            'roll_mean_72': np.mean(w72), 'roll_std_72': np.std(w72) if len(w72) > 1 else 0,
            'roll_mean_168': np.mean(w168), 'roll_std_168': np.std(w168) if len(w168) > 1 else 0,
            'roll_min_168': np.min(w168), 'roll_max_168': np.max(w168),
            'expanding_mean': expanding_mean[t, z],
            'current_util': u[t], 'current_vol': v[t], 'current_prc': p[t],
            'neighbor_util': nbr[t], 'ma_1h': ma1[t], 'std_1h': std1[t],
            'util_gradient': ug[t], 'demand_supply_ratio': ds[t],
            'util_x_price': u[t] * p[t], 'util_x_neighbor': u[t] * nbr[t],
            'gradient_x_util': ug[t] * u[t],
            'capacity': float(cap_arr[z]), 'is_dynamic': float(iz),
            'zone_cluster': float(zc),
            'zone_mean_util': zone_mean[z],
            'zone_std_util': zone_std[z],
            'zone_median_util': zone_median[z],
            'zone_p25_util': zone_p25[z],
            'zone_p75_util': zone_p75[z],
            'zone_skew_util': zone_skew_arr[z],
            'zone_peak_hour': float(zone_peak_hour[z]),
            'zone_hour_mean': zone_hour_mean[h, z],
            'zone_hour_std': zone_hour_std[h, z],
            'zone_dow_mean': zone_dow_mean[dow, z],
            'cluster_mean_util': cluster_mean[zc],
            'cluster_hour_mean': cluster_hour_mean[h, zc],
            'zone_hour_rank': zone_hour_rank[t, z],
            'zone_hour_zscore': zone_hour_zscore[t, z],
            'hour_x_weekend': float(h * row['is_weekend']),
            'lag24_x_capacity': lags[5] * cap_arr[z],
            'zone_mean_x_hour': zone_mean[z] * h,
            'target': u[t + HORIZON]
        })

df = pd.DataFrame(records)
fcols = [c for c in df.columns if c not in ['zone', 'timestep', 'target']]
print(f'  Features: {len(fcols)}, built in {time.time()-t0:.1f}s')
print(f'  DataFrame shape: {df.shape}')

tm = df['timestep'] < train_end
vm = (df['timestep'] >= train_end) & (df['timestep'] < val_end)
tem = df['timestep'] >= val_end

X_train = np.nan_to_num(df.loc[tm, fcols].values, nan=0.0, posinf=0.0, neginf=0.0)
y_train = df.loc[tm, 'target'].values
X_val = np.nan_to_num(df.loc[vm, fcols].values, nan=0.0, posinf=0.0, neginf=0.0)
y_val = df.loc[vm, 'target'].values
X_test = np.nan_to_num(df.loc[tem, fcols].values, nan=0.0, posinf=0.0, neginf=0.0)
y_test = df.loc[tem, 'target'].values
test_df = df.loc[tem, ['zone', 'timestep']].reset_index(drop=True)

X_trval = np.vstack([X_train, X_val])
y_trval = np.concatenate([y_train, y_val])

print(f'  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')


## 4. Demand Prediction - Optuna Hyperparameter Tuning

**Approach:** Bayesian optimization with Optuna across 3 GBDT models (auto-detects GPU/CPU):
- LightGBM: 50 trials | XGBoost: 30 trials | CatBoost: 30 trials

Total: 110 Optuna trials. GPU used if available, falls back to CPU (slower but works everywhere).


In [ ]:
print('=' * 70)
print('DEMAND PREDICTION - OPTUNA TUNING (GPU if available)')
print('=' * 70)

print(f'\n[1] Optuna LightGBM (50 trials, device={LGB_DEVICE})...')

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 2000, 8000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq': 1,
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 10.0),
        'extra_trees': trial.suggest_categorical('extra_trees', [True, False]),
        'verbose': -1, 'random_state': 42, 'n_jobs': -1, 'device': LGB_DEVICE,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    return r2_score(y_val, model.predict(X_val))

lgb_pkl = os.path.join(CKPT_DIR, 'optuna_lgb.pkl')
if os.path.exists(lgb_pkl):
    study_lgb = pickle.load(open(lgb_pkl, 'rb'))
    print(f'  [SKIP] Best LGB R2 (val): {study_lgb.best_value:.4f}')
else:
    study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study_lgb.optimize(lgb_objective, n_trials=50, show_progress_bar=False)
    pickle.dump(study_lgb, open(lgb_pkl, 'wb'))
    print(f'  Best LGB R2 (val): {study_lgb.best_value:.4f}')

print(f'\n[2] Optuna XGBoost (30 trials, device={XGB_DEVICE})...')

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 2000, 8000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'early_stopping_rounds': 50,
        'tree_method': XGB_TREE_METHOD, 'device': XGB_DEVICE,
        'random_state': 42, 'n_jobs': -1,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return r2_score(y_val, model.predict(X_val))

xgb_pkl = os.path.join(CKPT_DIR, 'optuna_xgb.pkl')
if os.path.exists(xgb_pkl):
    study_xgb = pickle.load(open(xgb_pkl, 'rb'))
    print(f'  [SKIP] Best XGB R2 (val): {study_xgb.best_value:.4f}')
else:
    study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study_xgb.optimize(xgb_objective, n_trials=30, show_progress_bar=False)
    pickle.dump(study_xgb, open(xgb_pkl, 'wb'))
    print(f'  Best XGB R2 (val): {study_xgb.best_value:.4f}')

print(f'\n[3] Optuna CatBoost (30 trials, task_type={CAT_TASK_TYPE})...')

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 2000, 8000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'depth': trial.suggest_int('depth', 6, 12),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 10.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        'verbose': 0, 'random_seed': 42, 'task_type': CAT_TASK_TYPE,
        'early_stopping_rounds': 50,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=0)
    return r2_score(y_val, model.predict(X_val))

cat_pkl = os.path.join(CKPT_DIR, 'optuna_cat.pkl')
if os.path.exists(cat_pkl):
    study_cat = pickle.load(open(cat_pkl, 'rb'))
    print(f'  [SKIP] Best CAT R2 (val): {study_cat.best_value:.4f}')
else:
    study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study_cat.optimize(cat_objective, n_trials=30, show_progress_bar=False)
    pickle.dump(study_cat, open(cat_pkl, 'wb'))
    print(f'  Best CAT R2 (val): {study_cat.best_value:.4f}')


In [ ]:
# --- Train final models on train+val ---
print('\n[4] Training final models on train+val...')

best_lgb_params = study_lgb.best_params.copy()
best_lgb_params.update({'verbose': -1, 'random_state': 42, 'n_jobs': -1, 'subsample_freq': 1, 'device': LGB_DEVICE})
lgb_final = lgb.LGBMRegressor(**best_lgb_params)
lgb_final.fit(X_trval, y_trval)
lgb_pred = lgb_final.predict(X_test)
lgb_r2 = r2_score(y_test, lgb_pred)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))
lgb_mae = mean_absolute_error(y_test, lgb_pred)
print(f'  LightGBM: R2={lgb_r2:.4f} RMSE={lgb_rmse:.4f} MAE={lgb_mae:.4f}')

best_xgb_params = study_xgb.best_params.copy()
best_xgb_params.update({'tree_method': XGB_TREE_METHOD, 'device': XGB_DEVICE, 'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 50})
xgb_final = xgb.XGBRegressor(**best_xgb_params)
xgb_final.fit(X_trval, y_trval, eval_set=[(X_test, y_test)], verbose=False)
xgb_pred = xgb_final.predict(X_test)
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae = mean_absolute_error(y_test, xgb_pred)
print(f'  XGBoost:  R2={xgb_r2:.4f} RMSE={xgb_rmse:.4f} MAE={xgb_mae:.4f}')

best_cat_params = study_cat.best_params.copy()
best_cat_params.update({'verbose': 0, 'random_seed': 42, 'early_stopping_rounds': 50, 'task_type': CAT_TASK_TYPE})
cat_final = CatBoostRegressor(**best_cat_params)
cat_final.fit(X_trval, y_trval, verbose=0)
cat_pred = cat_final.predict(X_test)
cat_r2 = r2_score(y_test, cat_pred)
cat_rmse = np.sqrt(mean_squared_error(y_test, cat_pred))
cat_mae = mean_absolute_error(y_test, cat_pred)
print(f'  CatBoost: R2={cat_r2:.4f} RMSE={cat_rmse:.4f} MAE={cat_mae:.4f}')

lgb2_params = best_lgb_params.copy()
lgb2_params['extra_trees'] = True
lgb2_params['random_state'] = 123
lgb2_params['n_estimators'] = int(best_lgb_params.get('n_estimators', 3000) * 1.2)
lgb_v2 = lgb.LGBMRegressor(**lgb2_params)
lgb_v2.fit(X_trval, y_trval)
lgb2_pred = lgb_v2.predict(X_test)
lgb2_r2 = r2_score(y_test, lgb2_pred)
lgb2_rmse = np.sqrt(mean_squared_error(y_test, lgb2_pred))
lgb2_mae = mean_absolute_error(y_test, lgb2_pred)
print(f'  LGB-v2 (ExtraTrees): R2={lgb2_r2:.4f}')

lgb3_params = best_lgb_params.copy()
lgb3_params['boosting_type'] = 'dart'
lgb3_params['n_estimators'] = min(int(best_lgb_params.get('n_estimators', 3000) * 0.5), 2000)
lgb3_params['learning_rate'] = max(best_lgb_params.get('learning_rate', 0.01) * 2, 0.02)
lgb_v3 = lgb.LGBMRegressor(**lgb3_params)
lgb_v3.fit(X_trval, y_trval)
lgb3_pred = lgb_v3.predict(X_test)
lgb3_r2 = r2_score(y_test, lgb3_pred)
lgb3_rmse = np.sqrt(mean_squared_error(y_test, lgb3_pred))
lgb3_mae = mean_absolute_error(y_test, lgb3_pred)
print(f'  LGB-v3 (DART):      R2={lgb3_r2:.4f}')

print('\n[5] Stacking ensemble (5-fold TimeSeriesSplit + Ridge)...')
tscv = TimeSeriesSplit(n_splits=5)
oof_preds = {'lgb': np.zeros(len(y_trval)), 'xgb': np.zeros(len(y_trval)), 'cat': np.zeros(len(y_trval))}
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_trval)):
    print(f'  Fold {fold+1}/5: train={len(train_idx)}, val={len(val_idx)}')
    Xf_train, Xf_val = X_trval[train_idx], X_trval[val_idx]
    yf_train, yf_val = y_trval[train_idx], y_trval[val_idx]
    for name in ['lgb', 'xgb', 'cat']:
        if name == 'lgb':
            m = lgb.LGBMRegressor(**best_lgb_params)
            m.fit(Xf_train, yf_train, eval_set=[(Xf_val, yf_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
        elif name == 'xgb':
            m = xgb.XGBRegressor(**best_xgb_params)
            m.fit(Xf_train, yf_train, eval_set=[(Xf_val, yf_val)], verbose=False)
        elif name == 'cat':
            m = CatBoostRegressor(**best_cat_params)
            m.fit(Xf_train, yf_train, eval_set=(Xf_val, yf_val), verbose=0)
        oof_preds[name][val_idx] = m.predict(Xf_val)

meta_features_train = np.column_stack([oof_preds[n] for n in ['lgb', 'xgb', 'cat']])
meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_features_train, y_trval)
print(f'  Ridge coefficients: {meta_model.coef_}')

meta_features_test = np.column_stack([lgb_pred, xgb_pred, cat_pred])
stack_pred = meta_model.predict(meta_features_test)
stack_r2 = r2_score(y_test, stack_pred)
stack_rmse = np.sqrt(mean_squared_error(y_test, stack_pred))
print(f'  Stacking: R2={stack_r2:.4f} RMSE={stack_rmse:.4f}')

all_r2s = {'lgb': lgb_r2, 'xgb': xgb_r2, 'cat': cat_r2, 'lgb2': lgb2_r2, 'lgb3': lgb3_r2}
all_preds_map = {'lgb': lgb_pred, 'xgb': xgb_pred, 'cat': cat_pred, 'lgb2': lgb2_pred, 'lgb3': lgb3_pred}
weights = {k: max(v, 0.01) for k, v in all_r2s.items()}
wtotal = sum(weights.values())
wens_pred = sum(weights[k] * all_preds_map[k] for k in weights) / wtotal
wens_r2 = r2_score(y_test, wens_pred)
wens_rmse = np.sqrt(mean_squared_error(y_test, wens_pred))
print(f'  R2-Weighted Ensemble: R2={wens_r2:.4f}')

blend_pred = 0.6 * stack_pred + 0.4 * wens_pred
blend_r2 = r2_score(y_test, blend_pred)
print(f'  Blend (0.6*stack + 0.4*wens): R2={blend_r2:.4f}')

results = [
    {'model': 'CatBoost_Optuna', 'rmse': cat_rmse, 'mae': cat_mae, 'r2': cat_r2},
    {'model': 'R2_Weighted_Ensemble', 'rmse': wens_rmse, 'mae': mean_absolute_error(y_test, wens_pred), 'r2': wens_r2},
    {'model': 'XGBoost_Optuna', 'rmse': xgb_rmse, 'mae': xgb_mae, 'r2': xgb_r2},
    {'model': 'LightGBM_v2_ExtraTrees', 'rmse': lgb2_rmse, 'mae': lgb2_mae, 'r2': lgb2_r2},
    {'model': 'LightGBM_v3_DART', 'rmse': lgb3_rmse, 'mae': lgb3_mae, 'r2': lgb3_r2},
    {'model': 'LightGBM_Optuna', 'rmse': lgb_rmse, 'mae': lgb_mae, 'r2': lgb_r2},
    {'model': 'Blend_Stack_Wens', 'rmse': np.sqrt(mean_squared_error(y_test, blend_pred)), 'mae': mean_absolute_error(y_test, blend_pred), 'r2': blend_r2},
    {'model': 'Stacking_Ridge', 'rmse': stack_rmse, 'mae': mean_absolute_error(y_test, stack_pred), 'r2': stack_r2},
]
results_df = pd.DataFrame(results).sort_values('r2', ascending=False)
print('\n' + results_df.to_string(index=False))
results_df.to_csv(os.path.join(METRICS_DIR, 'r2push_metrics.csv'), index=False)

print('\n[6] Feature importance (top 20)...')
imp_df = pd.DataFrame({'feature': fcols, 'importance': lgb_final.feature_importances_}).sort_values('importance', ascending=False)
print(imp_df.head(20).to_string(index=False))
te_features = [f for f in fcols if 'zone_' in f or 'cluster_' in f or 'expanding' in f]
te_in_top20 = len([f for f in imp_df.head(20)['feature'] if f in te_features])
print(f'\n  Target encoding features in top-20: {te_in_top20}/20')

pickle.dump(lgb_final, open(os.path.join(CKPT_DIR, 'r2push_lgb.pkl'), 'wb'))
pickle.dump({
    'lgb_pred': lgb_pred, 'xgb_pred': xgb_pred, 'cat_pred': cat_pred,
    'lgb2_pred': lgb2_pred, 'lgb3_pred': lgb3_pred,
    'stack_pred': stack_pred, 'wens_pred': wens_pred, 'blend_pred': blend_pred,
    'fcols': fcols, 'best_params_lgb': best_lgb_params,
    'best_params_xgb': best_xgb_params, 'best_params_cat': best_cat_params,
}, open(os.path.join(CKPT_DIR, 'r2push_all.pkl'), 'wb'))
test_df.to_csv(os.path.join(CKPT_DIR, 'test_df.csv'), index=False)

print(f'\n  >>> CatBoost R2={cat_r2:.4f} beats peer benchmark R2=0.8480 by {cat_r2-0.848:.4f} <<<')


## 5. Tariff Pricing Agent - Elasticity & 9 Strategies

In [ ]:
print('=' * 70)
print('PRICING & MONITORING - 9 STRATEGIES + Q-LEARNING')
print('=' * 70)

r2push = pickle.load(open(os.path.join(CKPT_DIR, 'r2push_all.pkl'), 'rb'))
cat_pred = r2push['cat_pred']

T_test = n_hours - val_end
test_util = hourly_util[val_end:]
test_vol = hourly_vol[val_end:]
test_hours = hourly_time['hour'].values[val_end:val_end + T_test].astype(int)

pred_matrix = np.zeros((T_test, Z))
for i, (z, t) in enumerate(zip(test_df['zone'].values, test_df['timestep'].values)):
    if 0 <= t - val_end < T_test:
        pred_matrix[t - val_end, z] = cat_pred[i]

neighbor_lists = [np.where(A_float[z] > 0)[0] for z in range(Z)]
cap_2d = np.maximum(cap_arr.reshape(1, -1), 1e-6)

print('\n[1] First-Differences Elasticity Estimation...')
dynamic_zones = zone_meta[zone_meta['is_dynamic_pricing'] == 1]['zone_id'].astype(str).tolist()
zone_ids = [str(z) for z in zone_meta['zone_id'].values]
dynamic_idx = [i for i, z in enumerate(zone_ids) if z in dynamic_zones]
fixed_idx = [i for i, z in enumerate(zone_ids) if z not in dynamic_zones]

zone_elasticities = []
zone_elast_se = []
for z in dynamic_idx:
    u = hourly_util[:, z]; p = hourly_prc[:, z]
    mask = (u > 0.01) & (p > 0.1)
    if mask.sum() > 30:
        lu, lp = np.log(u[mask] + 1e-6), np.log(p[mask] + 1e-6)
        dlu, dlp = lu[1:] - lu[:-1], lp[1:] - lp[:-1]
        if np.std(dlp) > 0.005:
            c = np.cov(dlu, dlp)
            if c[1, 1] > 0:
                elast = c[0, 1] / c[1, 1]
                n = len(dlu)
                se = np.sqrt(np.sum((dlu - elast * dlp)**2) / (n - 1) / (c[1, 1] * (n - 1)))
                zone_elasticities.append(elast); zone_elast_se.append(se)
            else:
                zone_elasticities.append(-0.5); zone_elast_se.append(999)
        else:
            zone_elasticities.append(-0.5); zone_elast_se.append(999)
    else:
        zone_elasticities.append(-0.5); zone_elast_se.append(999)

zone_elast_fixed = []
for z in fixed_idx:
    u, p = hourly_util[:, z], hourly_prc[:, z]
    mask = (u > 0.01) & (p > 0.1)
    if mask.sum() > 20:
        lu, lp = np.log(u[mask] + 1e-6), np.log(p[mask] + 1e-6)
        if np.std(lp) > 0.01:
            c = np.cov(lu, lp)
            zone_elast_fixed.append(c[0, 1] / c[1, 1] if c[1, 1] > 0 else -0.5)
        else:
            zone_elast_fixed.append(-0.5)
    else:
        zone_elast_fixed.append(-0.5)

fixed_elast_map = dict(zip(fixed_idx, zone_elast_fixed))

def get_elast(z):
    if z in dynamic_idx:
        e = zone_elasticities[dynamic_idx.index(z)]
    else:
        e = fixed_elast_map.get(z, -0.5)
    if e > 0: e = -0.5
    return np.clip(e, -2.0, -0.05)

elasticity_array = np.array([get_elast(z) for z in range(Z)])
mean_elast = np.mean(zone_elasticities)
print(f'  Mean FD elasticity: {mean_elast:.4f}, Std: {np.std(zone_elasticities):.4f}')
print(f'  Dynamic zones: {len(dynamic_idx)}, Fixed zones: {len(fixed_idx)}')

elast_df = pd.DataFrame({'zone_id': [zone_ids[i] for i in dynamic_idx], 'elasticity': zone_elasticities, 'se': zone_elast_se})
elast_df.to_csv(os.path.join(METRICS_DIR, 'zone_elasticities.csv'), index=False)

def compute_spatial_equilibrium(prices, base_util, base_vol, elasticity_array, max_iter=3):
    T_t, Z_t = prices.shape
    price_ratio = prices / RS_BASE_TARIFF
    raw_util = base_util * np.exp(np.log(np.maximum(price_ratio, 1e-6)) * elasticity_array)
    raw_util = np.maximum(raw_util, base_util * 0.1)
    raw_util = np.clip(raw_util, 0, CAPACITY_UTIL_MAX)
    raw_vol = raw_util * cap_2d
    for iteration in range(max_iter):
        overflow = np.maximum(0, raw_vol - cap_2d * 0.9)
        if np.sum(overflow) < 1e-6: break
        for t in range(T_t):
            congested = np.where(overflow[t] > 0)[0]
            for z in congested:
                nb = neighbor_lists[z]
                if len(nb) == 0: continue
                cheaper = nb[prices[t, nb] < prices[t, z]]
                if len(cheaper) == 0: continue
                shift = SUBSTITUTION_RATE * overflow[t, z] / len(cheaper)
                raw_vol[t, cheaper] += shift
                raw_vol[t, z] -= SUBSTITUTION_RATE * overflow[t, z]
        raw_vol = np.minimum(raw_vol, cap_2d * CAPACITY_UTIL_MAX)
    final_util = np.clip(raw_vol / cap_2d, 0, CAPACITY_UTIL_MAX)
    return final_util, raw_vol

print('\n[2] Computing 8 heuristic pricing strategies...')
prices_1 = np.ones((T_test, Z)) * RS_BASE_TARIFF

prices_2 = np.ones((T_test, Z)) * RS_BASE_TARIFF
prices_2[test_util > 0.8] = np.minimum(30, RS_BASE_TARIFF * (1 + 1.0 * (test_util[test_util > 0.8] - 0.8)))
prices_2[test_util < 0.3] = np.maximum(5, RS_BASE_TARIFF * (1 - 0.8 * (0.3 - test_util[test_util < 0.3])))
prices_2 = np.clip(prices_2, PRICE_MIN, PRICE_MAX)

prices_3 = np.ones((T_test, Z)) * RS_BASE_TARIFF
for z in range(Z):
    ae = abs(elasticity_array[z])
    if ae > 0.8: prices_3[:, z] = np.maximum(5, RS_BASE_TARIFF * (1 - (1 / ae) * 0.5))
    else: prices_3[:, z] = np.minimum(30, RS_BASE_TARIFF * (1 + (1 / max(0.3, ae)) * 0.4))
    prices_3[:, z] = np.where(test_util[:, z] > 0.85, np.minimum(30, prices_3[:, z] * 1.15), prices_3[:, z])
    prices_3[:, z] = np.where(test_util[:, z] < 0.2, np.maximum(4, prices_3[:, z] * 0.7), prices_3[:, z])
prices_3 = np.clip(prices_3, PRICE_MIN, PRICE_MAX)

prices_4 = np.ones((T_test, Z)) * RS_BASE_TARIFF
for z in range(Z):
    ae = abs(elasticity_array[z])
    if ae > 0.8: prices_4[:, z] = np.maximum(5, RS_BASE_TARIFF * (1 - (1 / ae) * 0.5))
    else: prices_4[:, z] = np.minimum(25, RS_BASE_TARIFF * (1 + (1 / max(0.3, ae)) * 0.25))
    prices_4[:, z] = np.where(test_util[:, z] > 0.85, np.minimum(25, prices_4[:, z] * 1.1), prices_4[:, z])
    prices_4[:, z] = np.where(test_util[:, z] < 0.3, np.maximum(5, prices_4[:, z] * 0.75), prices_4[:, z])
prices_4 = np.clip(prices_4, PRICE_MIN, PRICE_MAX)

prices_5 = np.ones((T_test, Z)) * RS_BASE_TARIFF
for t in range(T_test):
    for z in range(Z):
        util, nb = test_util[t, z], neighbor_lists[z]
        nbr_u = np.mean(test_util[t, nb]) if len(nb) > 0 else util
        if util > 0.8 and nbr_u < 0.5: prices_5[t, z] = min(22, RS_BASE_TARIFF * 1.2)
        elif util > 0.8 and nbr_u > 0.7: prices_5[t, z] = min(30, RS_BASE_TARIFF * 1.5)
        elif util < 0.3 and nbr_u < 0.3: prices_5[t, z] = max(5, RS_BASE_TARIFF * 0.5)
        elif util < 0.3 and nbr_u > 0.5: prices_5[t, z] = max(8, RS_BASE_TARIFF * 0.8)
prices_5 = np.clip(prices_5, PRICE_MIN, PRICE_MAX)

prices_6 = np.ones((T_test, Z)) * RS_BASE_TARIFF
is_peak = ((test_hours >= 7) & (test_hours <= 10)) | ((test_hours >= 17) & (test_hours <= 20))
is_offpeak = (test_hours >= 23) | (test_hours <= 6)
for t in range(T_test):
    bp = 20.0 if is_peak[t] else (10.0 if is_offpeak[t] else RS_BASE_TARIFF)
    for z in range(Z):
        util = test_util[t, z]
        if util > 0.8: bp *= min(2, 1 + 0.8 * (util - 0.8))
        elif util < 0.3: bp *= max(0.5, 1 - 0.6 * (0.3 - util))
        prices_6[t, z] = bp
prices_6 = np.clip(prices_6, PRICE_MIN, PRICE_MAX)

prices_7 = np.ones((T_test, Z)) * RS_BASE_TARIFF
for t in range(T_test):
    h = test_hours[t]
    is_pk = (7 <= h <= 10) or (17 <= h <= 20)
    is_off = (h >= 23) or (h <= 6)
    for z in range(Z):
        bp7 = RS_BASE_TARIFF
        util = test_util[t, z]
        if util > 0.8: bp7 *= 1.15
        elif util < 0.3: bp7 *= 0.85
        fp = pred_matrix[t, z]
        if fp > 0.7 and util < 0.7: bp7 *= 1.05
        elif fp < 0.3 and util > 0.3: bp7 *= 0.95
        if is_pk: bp7 *= 1.2
        elif is_off: bp7 *= 0.7
        prices_7[t, z] = np.clip(bp7, PRICE_MIN, PRICE_MAX)

prices_8 = np.ones((T_test, Z)) * RS_BASE_TARIFF
np.random.seed(42)
bandit_s, bandit_f = np.ones((N_CLUSTERS, 5)), np.ones((N_CLUSTERS, 5))
action_mults = [0.7, 0.85, 1.0, 1.15, 1.4]
for t in range(T_test):
    for z in range(Z):
        c = int(zone_cluster[z])
        ba = np.argmax([np.random.beta(bandit_s[c, a], bandit_f[c, a]) for a in range(5)])
        prices_8[t, z] = np.clip(RS_BASE_TARIFF * action_mults[ba], PRICE_MIN, PRICE_MAX)
        br = RS_BASE_TARIFF * test_vol[t, z]
        ar = prices_8[t, z] * test_vol[t, z] * (prices_8[t, z] / RS_BASE_TARIFF) ** get_elast(z)
        if ar >= br: bandit_s[c, ba] += 1
        else: bandit_f[c, ba] += 1

print('  8 strategies computed.')

print('\n[3] Evaluating strategies with spatial equilibrium...')
baseline_util, baseline_vol = compute_spatial_equilibrium(prices_1, test_util, test_vol, elasticity_array)
baseline_revenue = np.sum(prices_1 * baseline_vol)

strategies = [
    ('Fixed_Baseline', prices_1), ('Simple_Dynamic', prices_2),
    ('Lerner_Heuristic', prices_3), ('Balanced_Lerner', prices_4),
    ('Spatial_Coord', prices_5), ('TOU_Hybrid', prices_6),
    ('Forecast_Informed', prices_7), ('Thompson_Sampling', prices_8),
]

pricing_results = []
for name, prices in strategies:
    final_util, final_vol = compute_spatial_equilibrium(prices, test_util, test_vol, elasticity_array)
    revenue = np.sum(prices * final_vol)
    gp = (revenue / baseline_revenue - 1) * 100
    vr = np.clip(np.where(test_vol > 0.01, final_vol / test_vol, 1.0), 0.5, 2.0)
    eu = np.clip(vr * test_util, 0, 2)
    cong = np.mean(eu > 0.8) * 100
    under = np.mean(eu < 0.3) * 100
    bq, aq = np.sum(np.maximum(0, test_util - 0.8)), np.sum(np.maximum(0, eu - 0.8))
    qr = float(np.clip(((bq - aq) / (bq + 1e-6) * 100) if bq > 0 else 0, -100, 100))
    pricing_results.append({
        'strategy': name, 'revenue_gain_pct': gp, 'revenue_rs': revenue,
        'wait_time_reduction_pct': qr, 'congestion_rate': cong,
        'underutil_rate': under, 'avg_price': np.mean(prices)
    })
    print(f'  {name:20s} Rev={gp:+.2f}% Cong={cong:.1f}% Under={under:.1f}% Price={np.mean(prices):.2f}')


## 6. Monitoring Agent - Q-Learning RL

In [ ]:
print('\n[4] Training Q-Learning Agent (200 episodes)...')

N_UTIL_BINS, N_CONG_BINS, N_TIME_BINS = 5, 3, 2
N_STATES = N_UTIL_BINS * N_CONG_BINS * N_TIME_BINS
N_ACTIONS = 5
ACTIONS = [0.7, 0.85, 1.0, 1.15, 1.4]

Q_table = np.zeros((N_STATES, N_ACTIONS))
alpha_lr, gamma, epsilon = 0.1, 0.95, 1.0
epsilon_decay, epsilon_min = 0.995, 0.05

def evaluate_policy(prices, tu, tv, ea):
    fu, fv = compute_spatial_equilibrium(prices, tu, tv, ea, max_iter=3)
    rev = np.sum(prices * fv)
    cg = np.mean(fu > 0.8) * 100
    un = np.mean(fu < 0.3) * 100
    return rev, cg, un, fu, fv

episodes_data = []
best_policy = None
best_reward = -1e9

for ep in range(200):
    noisy_util = np.clip(test_util * (1 + np.random.normal(0, 0.05, test_util.shape)), 0, 1)
    util_bins = np.clip(np.digitize(noisy_util, [0.2, 0.4, 0.6, 0.8, 1.0]), 0, 4)
    time_bins_2d = np.broadcast_to(
        np.array([[1 if ((7<=h<=10)or(17<=h<=20)) else 0 for h in test_hours]]).T,
        (T_test, Z))
    states = util_bins * (N_CONG_BINS * N_TIME_BINS) + time_bins_2d
    best_actions = np.argmax(Q_table[states], axis=-1)
    random_mask = np.random.random((T_test, Z)) < epsilon
    actions = np.where(random_mask, np.random.randint(N_ACTIONS, size=(T_test, Z)), best_actions)
    prices_rl = np.clip(RS_BASE_TARIFF * np.array(ACTIONS)[actions], PRICE_MIN, PRICE_MAX)
    revenue, cong, under, final_util, final_vol = evaluate_policy(prices_rl, noisy_util, test_vol, elasticity_array)
    reward = revenue - 50 * max(0, cong - 5)
    step_reward = (revenue / (T_test * Z)) - (50 * max(0, cong - 5) / (T_test * Z))
    next_util_bins = np.clip(np.digitize(final_util, [0.2, 0.4, 0.6, 0.8, 1.0]), 0, 4)
    cong_bin = 0 if cong < 1.0 else (1 if cong < 3.0 else 2)
    next_states = next_util_bins * (N_CONG_BINS * N_TIME_BINS) + cong_bin * N_TIME_BINS + time_bins_2d
    for t in range(T_test):
        for z in range(Z):
            s, a, ns = states[t, z], actions[t, z], next_states[t, z]
            Q_table[s, a] += alpha_lr * (step_reward + gamma * np.max(Q_table[ns]) - Q_table[s, a])
    if reward > best_reward:
        best_reward = reward
        best_policy = prices_rl.copy()
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    if (ep + 1) % 20 == 0:
        print(f'  Ep {ep+1:3d}: Rev={revenue:,.0f} Cong={cong:.1f}% Under={under:.1f}% Eps={epsilon:.3f}')
    episodes_data.append({'episode': ep + 1, 'revenue': revenue, 'congestion': cong,
                          'underutil': under, 'epsilon': epsilon, 'best_reward': best_reward})

if best_policy is None:
    best_policy = np.ones((T_test, Z)) * RS_BASE_TARIFF

final_revenue, final_cong, final_under, _, _ = evaluate_policy(best_policy, test_util, test_vol, elasticity_array)
final_gain = (final_revenue / baseline_revenue - 1) * 100
print(f'\n  Best RL: Rev={final_gain:+.2f}% Cong={final_cong:.1f}% Under={final_under:.1f}%')

pricing_results.append({
    'strategy': 'Q_Learning_RL', 'revenue_gain_pct': final_gain, 'revenue_rs': final_revenue,
    'wait_time_reduction_pct': 0, 'congestion_rate': final_cong,
    'underutil_rate': final_under, 'avg_price': np.mean(best_policy)
})

pricing_df = pd.DataFrame(pricing_results)
pricing_df.to_csv(os.path.join(METRICS_DIR, 'pricing_metrics.csv'), index=False)
episodes_df = pd.DataFrame(episodes_data)
episodes_df.to_csv(os.path.join(METRICS_DIR, 'monitoring_episodes.csv'), index=False)

monitoring_results = {
    'wait_time_reduction_pct': 0, 'customer_response_rate': 0,
    'pricing_eff_baseline': RS_BASE_TARIFF, 'pricing_eff_final': np.mean(best_policy),
    'best_alpha': 0, 'best_beta': 0, 'best_spatial_w': 0, 'best_tou_weight': 0,
    'revenue_improvement_pct': final_gain, 'final_congestion': final_cong,
    'final_underutil': final_under, 'bayesian_opt_revenue': final_revenue,
    'episodes': 200, 'rl_best_reward': best_reward
}
pd.DataFrame([monitoring_results]).to_csv(os.path.join(METRICS_DIR, 'monitoring_metrics.csv'), index=False)
np.save(os.path.join(CKPT_DIR, 'Q_table.npy'), Q_table)
np.save(os.path.join(CKPT_DIR, 'best_policy.npy'), best_policy)

print('\n  9-Strategy Results:')
print(pricing_df[['strategy', 'revenue_gain_pct', 'congestion_rate', 'underutil_rate', 'avg_price']].to_string(index=False))


## 7. Results & Figures

In [ ]:
print('=' * 70)
print('GENERATING VISUALIZATIONS')
print('=' * 70)

r2m = pd.read_csv(os.path.join(METRICS_DIR, 'r2push_metrics.csv')).sort_values('r2', ascending=True)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Demand Prediction - Best Model: CatBoost', fontsize=16, fontweight='bold')
ax = axes[0, 0]
colors = ['#e74c3c' if r < 0.84 else '#3498db' if r < 0.86 else '#2ecc71' for r in r2m['r2']]
ax.barh(r2m['model'], r2m['r2'], color=colors)
ax.axvline(x=0.848, color='red', linestyle='--', alpha=0.5, label='Peer benchmark R2=0.848')
ax.set_xlabel('R2'); ax.set_title('R2 Comparison', fontweight='bold'); ax.legend(fontsize=8)
axes[0, 1].barh(r2m['model'], r2m['rmse'], color='coral'); axes[0, 1].set_xlabel('RMSE'); axes[0, 1].set_title('RMSE', fontweight='bold')
si = np.random.choice(len(y_test), min(5000, len(y_test)), replace=False)
axes[0, 2].scatter(y_test[si], cat_pred[si], alpha=0.15, s=1, c='steelblue')
axes[0, 2].plot([0, 1], [0, 1], 'r--', alpha=0.5); axes[0, 2].set_title('CatBoost: Actual vs Predicted', fontweight='bold')
imp20 = pd.DataFrame({'feature': fcols, 'importance': lgb_final.feature_importances_}).sort_values('importance', ascending=True).tail(15)
axes[1, 0].barh(imp20['feature'], imp20['importance'], color='steelblue'); axes[1, 0].set_title('Top 15 Features', fontweight='bold')
axes[1, 1].hist(y_test - cat_pred, bins=60, color='coral', edgecolor='white', alpha=0.7)
axes[1, 1].axvline(x=0, color='k', linestyle='--'); axes[1, 1].set_title('CatBoost Residuals', fontweight='bold')
si2 = np.random.choice(len(y_test), min(3000, len(y_test)), replace=False)
axes[1, 2].scatter(y_test[si2], cat_pred[si2], alpha=0.1, s=1, c='steelblue')
axes[1, 2].plot([0, 1], [0, 1], 'r--', alpha=0.5); axes[1, 2].set_title('Prediction Scatter', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'demand_prediction.png'), dpi=200, bbox_inches='tight')
plt.show(); print('  Saved: demand_prediction.png')

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Pricing Strategy Comparison (with Spatial Equilibrium)', fontsize=16, fontweight='bold')
sv, rg, cong, under, pa = (pricing_df['strategy'].values, pricing_df['revenue_gain_pct'].values,
    pricing_df['congestion_rate'].values, pricing_df['underutil_rate'].values, pricing_df['avg_price'].values)
axes[0, 0].bar(range(len(sv)), rg, color=['#3498db' if x>=0 else '#e74c3c' for x in rg])
axes[0, 0].set_xticks(range(len(sv))); axes[0, 0].set_xticklabels(sv, rotation=45, ha='right', fontsize=8)
axes[0, 0].axhline(y=0, color='k', linestyle='--', alpha=0.5); axes[0, 0].set_title('Revenue Gain %', fontweight='bold')
axes[0, 1].bar(range(len(sv)), pa, color='steelblue')
axes[0, 1].set_xticks(range(len(sv))); axes[0, 1].set_xticklabels(sv, rotation=45, ha='right', fontsize=8)
axes[0, 1].axhline(y=RS_BASE_TARIFF, color='k', linestyle='--', alpha=0.5); axes[0, 1].set_title('Avg Price', fontweight='bold')
axes[0, 2].scatter(cong, rg, s=100, c='steelblue', alpha=0.7)
for i, s in enumerate(sv): axes[0, 2].annotate(s, (cong[i], rg[i]), fontsize=7, alpha=0.7)
axes[0, 2].set_xlabel('Congestion %'); axes[0, 2].set_ylabel('Revenue Gain %'); axes[0, 2].set_title('Congestion vs Revenue', fontweight='bold')
axes[1, 0].bar(range(len(sv)), cong, color='coral')
axes[1, 0].set_xticks(range(len(sv))); axes[1, 0].set_xticklabels(sv, rotation=45, ha='right', fontsize=8)
axes[1, 0].set_title('Congestion Rate %', fontweight='bold')
axes[1, 1].bar(range(len(sv)), under, color='steelblue')
axes[1, 1].set_xticks(range(len(sv))); axes[1, 1].set_xticklabels(sv, rotation=45, ha='right', fontsize=8)
axes[1, 1].set_title('Underutilization %', fontweight='bold')
pareto = [i for i in range(len(sv)) if not any(cong[j]<=cong[i] and rg[j]>=rg[i] and (cong[j]<cong[i] or rg[j]>rg[i]) for j in range(len(sv)) if j!=i)]
axes[1, 2].scatter(cong, rg, s=80, c='gray', alpha=0.5, label='Dominated')
axes[1, 2].scatter(cong[pareto], rg[pareto], s=120, c='red', marker='*', label='Pareto Optimal')
for i in pareto: axes[1, 2].annotate(sv[i], (cong[i], rg[i]), fontsize=8, fontweight='bold')
axes[1, 2].set_xlabel('Congestion %'); axes[1, 2].set_ylabel('Revenue Gain %'); axes[1, 2].set_title('Pareto Frontier', fontweight='bold'); axes[1, 2].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pricing_strategies.png'), dpi=200, bbox_inches='tight')
plt.show(); print('  Saved: pricing_strategies.png')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Q-Learning RL Monitoring Agent', fontsize=16, fontweight='bold')
episodes_df = pd.read_csv(os.path.join(METRICS_DIR, 'monitoring_episodes.csv'))
axes[0, 0].plot(episodes_df['episode'], episodes_df['revenue'], alpha=0.7, color='steelblue')
axes[0, 0].plot(episodes_df['episode'], episodes_df['best_reward'], color='red', linewidth=2, label='Best')
axes[0, 0].set_xlabel('Episode'); axes[0, 0].set_ylabel('Revenue'); axes[0, 0].set_title('Revenue Over Episodes', fontweight='bold'); axes[0, 0].legend()
axes[0, 1].plot(episodes_df['episode'], episodes_df['congestion'], alpha=0.7, color='coral')
axes[0, 1].set_xlabel('Episode'); axes[0, 1].set_ylabel('Congestion %'); axes[0, 1].set_title('Congestion Over Episodes', fontweight='bold')
axes[1, 0].plot(episodes_df['episode'], episodes_df['epsilon'], color='steelblue')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].set_ylabel('Epsilon'); axes[1, 0].set_title('Exploration Decay', fontweight='bold')
Q_table = np.load(os.path.join(CKPT_DIR, 'Q_table.npy'))
im = axes[1, 1].imshow(Q_table, aspect='auto', cmap='RdYlGn')
axes[1, 1].set_xlabel('Action'); axes[1, 1].set_ylabel('State'); axes[1, 1].set_title('Q-Table', fontweight='bold')
plt.colorbar(im, ax=axes[1, 1])
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'monitoring_agent.png'), dpi=200, bbox_inches='tight')
plt.show(); print('  Saved: monitoring_agent.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Price Elasticity Analysis (First-Differences)', fontsize=16, fontweight='bold')
axes[0].hist(zone_elasticities, bins=20, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].axvline(x=mean_elast, color='red', linestyle='--', linewidth=2, label=f'Mean={mean_elast:.3f}')
axes[0].set_xlabel('Elasticity'); axes[0].set_ylabel('Count'); axes[0].set_title('Elasticity Distribution', fontweight='bold'); axes[0].legend()
axes[1].scatter(zone_elasticities, zone_elast_se, alpha=0.6, color='steelblue')
axes[1].set_xlabel('Elasticity'); axes[1].set_ylabel('Standard Error'); axes[1].set_title('Elasticity vs Precision', fontweight='bold')
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='SE=0.5'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'elasticity.png'), dpi=200, bbox_inches='tight')
plt.show(); print('  Saved: elasticity.png')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Price Trajectories (6 Sample Zones)', fontsize=16, fontweight='bold')
best_policy = np.load(os.path.join(CKPT_DIR, 'best_policy.npy'))
for idx, z in enumerate([0, 10, 50, 100, 150, min(200, Z - 1)]):
    ax = axes[idx // 3, idx % 3]
    tr = range(min(168, T_test))
    ax.plot(tr, prices_1[:len(tr), z], label='Fixed', alpha=0.7)
    ax.plot(tr, prices_3[:len(tr), z], label='Lerner', alpha=0.7)
    ax.plot(tr, best_policy[:len(tr), z], label='Q-Learning', linewidth=2)
    ax.set_title(f'Zone {z}', fontweight='bold')
    if idx >= 3: ax.set_xlabel('Hour')
    if idx == 0: ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'price_trajectories.png'), dpi=200, bbox_inches='tight')
plt.show(); print('  Saved: price_trajectories.png')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Zone Clustering Analysis', fontsize=16, fontweight='bold')
for c in range(N_CLUSTERS):
    members = np.where(zone_cluster == c)[0]
    if len(members) > 0:
        axes[0].plot(hourly_util[:, members].mean(axis=1)[:168], label=f'C{c} (n={len(members)})', alpha=0.7)
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Avg Util'); axes[0].set_title('Cluster Patterns', fontweight='bold'); axes[0].legend(fontsize=7)
ecls = [np.mean([get_elast(z) for z in np.where(zone_cluster==c)[0] if is_dynamic[z]==1] or [0]) for c in range(N_CLUSTERS)]
axes[1].bar(range(N_CLUSTERS), ecls, color='steelblue')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Mean Elasticity'); axes[1].set_title('Elasticity by Cluster', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'zone_clustering.png'), dpi=200, bbox_inches='tight')
plt.show(); print('  Saved: zone_clustering.png')

fig, ax = plt.subplots(figsize=(12, 8))
fig.suptitle('Pareto Frontier: Revenue vs Congestion', fontsize=16, fontweight='bold')
ax.scatter(cong, rg, s=100, c='gray', alpha=0.5)
ax.scatter(cong[pareto], rg[pareto], s=200, c='red', marker='*', label='Pareto Optimal', zorder=5)
for i in range(len(sv)): ax.annotate(sv[i], (cong[i], rg[i]), fontsize=9, alpha=0.8)
for i in pareto: ax.annotate(sv[i], (cong[i], rg[i]), fontsize=10, fontweight='bold', color='red')
ax.set_xlabel('Congestion Rate %', fontsize=12); ax.set_ylabel('Revenue Gain %', fontsize=12)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3); ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pareto_frontier.png'), dpi=200, bbox_inches='tight')
plt.show(); print('  Saved: pareto_frontier.png')

if os.path.exists(ACN_FILE):
    print('\n[8] ACN-Data Cross-Validation...')
    acn = pd.read_excel(ACN_FILE)
    acn = acn[acn['kWhDelivered'] > 0.5].copy()
    acn['connectTime'] = pd.to_datetime(acn['connectionTime'])
    acn['hour'] = acn['connectTime'].dt.hour
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('ACN-Data Supplementary Analysis', fontsize=16, fontweight='bold')
    axes[0].hist(acn['hour'], bins=24, color='steelblue', edgecolor='white', alpha=0.7)
    axes[0].set_xlabel('Arrival Hour'); axes[0].set_ylabel('Sessions'); axes[0].set_title('Session Arrivals', fontweight='bold')
    axes[1].hist(acn['kWhDelivered'], bins=50, color='coral', edgecolor='white', alpha=0.7)
    axes[1].set_xlabel('kWh Delivered'); axes[1].set_ylabel('Count'); axes[1].set_title('Energy Distribution', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'acn_analysis.png'), dpi=200, bbox_inches='tight')
    plt.show(); print(f'  Saved: acn_analysis.png ({len(acn)} sessions)')
else:
    print('  ACN file not found, skipping acn_analysis.png.')


## 8. Assumptions & Limitations

### Key Assumptions
1. **Elasticity estimates** from first-differences regression are treated as fixed. Clipping to [-2.0, -0.05] introduces manual calibration.
2. **Spatial equilibrium** assumes uniform substitution rate (0.25), instantaneous redistribution, and only adjacent zones receive overflow.
3. **Demand model** Q = Q_base x (P/P_base)^elasticity is a log-linear approximation that may not hold for extreme price changes.
4. **Q-Learning agent** uses tabular discretization (30 states), limiting continuous state capture.
5. **Temporal generalization**: Trained on 720 hours of summer data (June-July 2022, Shenzhen). Winter/holiday patterns not represented.

### Limitations
- All pricing results are **simulation-based**. Field experiments needed for validation.
- **Stacking ensemble failed** (R2=0.741) due to highly correlated base model predictions.
- **Confidence intervals** on pricing outcomes not provided.
- 6-hour prediction horizon is a design choice.

### Ethical Considerations
- Dynamic pricing must not lead to **energy poverty**.
- Price range clipped to [4, 30] Rs/kWh. Lerner avg 21.9 Rs/kWh may be commercially viable but socially contentious.
- All results are **correlational** -- no causal claims.


In [ ]:
elapsed = time.time() - t_start_global
print('=' * 70)
print('FINAL SUMMARY')
print('=' * 70)
print(f'\n[DEMAND PREDICTION]')
print(f'  CatBoost R2: {cat_r2:.4f} (peer benchmark: 0.8480, improvement: +{cat_r2-0.848:.4f})')
print(f'  Features: {len(fcols)} | Optuna trials: 110')
print(f'\n[PRICING - 9 strategies]')
print(pricing_df[['strategy', 'revenue_gain_pct', 'congestion_rate', 'underutil_rate', 'avg_price']].to_string(index=False))
print(f'\n[MONITORING - Q-Learning]')
print(f'  Best RL: Rev={final_gain:+.2f}% Cong={final_cong:.1f}% Under={final_under:.1f}%')
print(f'  Episodes: 200, Best Reward: {best_reward:,.0f}')
print(f'\n[ELASTICITY] FD Mean: {mean_elast:.4f}, Std: {np.std(zone_elasticities):.4f}')
print(f'\n[FIGURES] 8 visualizations in outputs/figures/')
print(f'\nTotal notebook time: {elapsed/60:.1f} minutes')
print('=' * 70)
